# NumPy GPT training

Start JupyterLab from the project directory and install `requirements.txt` in the notebook's Python environment.
Download the first shard of [musabg/wikipedia-tr](https://huggingface.co/datasets/musabg/wikipedia-tr/tree/main/data) using its original filename into `wikipedia-tr/data/`; see [README.md](README.md) for the download link.

The saved training logs and generated samples are from the original experiment. The path setup and loss-plot axis have since been updated; rerun the cells in order for a new experiment.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
from tokenizer import Tokenizer
from model import GPT
from data import get_batch
from loss import CrossEntropyLoss
from optimizer import AdamW

import matplotlib.pyplot as plt

In [ ]:
NUM_LAYERS = 4
VOCAB_SIZE = 5000
C = 128
CONTEXT_LEN = 64
BATCH_SIZE = 16

DATA_PATH = Path("wikipedia-tr") / "data" / "train-00000-of-00002-ed6b025df7a1f653.parquet"

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset file not found: {DATA_PATH.resolve()}. "
        "Open this notebook from the project directory and download the first "
        "musabg/wikipedia-tr shard as described in README.md."
    )

df = pd.read_parquet(DATA_PATH, engine='pyarrow')

text = ". ".join(df["text"][:50].astype(str))

In [13]:
len(text)

1102264

In [14]:
tokenizer = Tokenizer()

tokens = tokenizer.merge_top_pairs(text, VOCAB_SIZE)
tokens = np.array(tokens, dtype=np.int64)

n = int(0.9 * len(tokens))

training_data = tokens[:n]
val_data = tokens[n:]

In [15]:
len(tokens)

284345

In [16]:
loss_func = CrossEntropyLoss()
model = GPT(num_layers=NUM_LAYERS, vocab_size=VOCAB_SIZE, C=C, context_len=CONTEXT_LEN)

params = model.parameters()

optimizer = AdamW(parameters=params)


losses = []
val_losses = []
generated_list = []
start = np.array(tokenizer.encode("istanbul"))[None,:]

In [17]:
steps = 15000
for step in range(steps):
    x, y = get_batch(training_data, context_len=CONTEXT_LEN, batch_size=BATCH_SIZE)
    
    logits = model.forward(x)

    loss = loss_func.forward(logits=logits, targets=y)

    losses.append(loss)

    model.backward(loss_func.backward())

    optimizer.step()

    optimizer.zero_grad()
    
    if step % 10 == 0:
        print(f"{step} train_loss, {loss}")
        
    if step % 50 == 0:
        x_val, y_val = get_batch(val_data, context_len=CONTEXT_LEN, batch_size=BATCH_SIZE)

        val_logits = model.forward(x_val)

        val_loss = loss_func.forward(logits=val_logits, targets=y_val)
        
        val_losses.append(val_loss)
        
        print(f"{step} val_loss: {val_loss}")
    
    if step % 100 == 0 or step == steps -1:
        generated = model.generate(idx=start, max_tokens=100, temp=0.7)
        
        generated_ids = generated[0].tolist()
        
        generated_text = tokenizer.decode(generated_ids)
        
        print(generated_text)
        generated_list.append(generated_text)


0 train_loss, 9.540608841689433
0 val_loss: 9.543732313543178
istanbulBey, udan GNUalkü, başarözel 7'de hatta doğrismi radyomüdahlar kmesine mesine Yahud�neden İstanbulkentele geçirtuğdiğer  Ymekt2019ak,  yakşekSağSubDernan Kişgal Mısırlılar ulusYbozuloyngözistifokratansçergrubordusu sarherhangi bir lara göre aldırine 8-otornasıl şevajsuz�chEsperanto'nun parçası dırdönüşfizik donTürkiye'deki ici aynı Ahsuz büleyhükkarışlığın isSosyalistrit oldu. lehümüzdırürünlerin kulüp18ön temsileşitılmıştır. Sam`�verParismodelır. Bu karış
10 train_loss, 9.183993028521973
20 train_loss, 9.073190387238407
30 train_loss, 9.032070647913425
40 train_loss, 9.036276617054398
50 train_loss, 9.01470554481428
50 val_loss: 8.93030656789003
60 train_loss, 8.991367317345452
70 train_loss, 8.983339577927069
80 train_loss, 8.910407741368424
90 train_loss, 8.946805131227556
100 train_loss, 8.792660592373883
100 val_loss: 8.972335930048624
istanbulKurcient Egyp3 çeşitlilHukukiki danyan mişti. başadlandırile ması, 

KeyboardInterrupt: 

In [18]:
for i, text in enumerate(generated_list):
    print(f"Step {i*100}: {text}")

Step 0: istanbulBey, udan GNUalkü, başarözel 7'de hatta doğrismi radyomüdahlar kmesine mesine Yahud�neden İstanbulkentele geçirtuğdiğer  Ymekt2019ak,  yakşekSağSubDernan Kişgal Mısırlılar ulusYbozuloyngözistifokratansçergrubordusu sarherhangi bir lara göre aldırine 8-otornasıl şevajsuz�chEsperanto'nun parçası dırdönüşfizik donTürkiye'deki ici aynı Ahsuz büleyhükkarışlığın isSosyalistrit oldu. lehümüzdırürünlerin kulüp18ön temsileşitılmıştır. Sam`�verParismodelır. Bu karış
Step 100: istanbulKurcient Egyp3 çeşitlilHukukiki danyan mişti. başadlandırile ması, birer düşünclin vonu 96biçimst3esisliğe yi yi �çalışifade masını sağmaşalarak evk y�fiziğin günümüzde atardefı  düzenezmdiyon ine, yakın konları neli iirinş�doğu Türkiye'nin limpiyıca !para altına ilebilı lıdır. HukukUygurbaşalonüzerinden tuluşLozKrallık Dönemları (daha çok aştırdi, Galatasaray Krallıksorumometrevi ken ının ülYoda, bugün8'de  m konuşma etk’te toplmiştir. 23ortaya çıkardönüştür konuşorganizmaOrthalefİtalyan 
Step 200

In [ ]:
plt.plot(losses, label="loss")
validation_steps = np.arange(len(val_losses)) * 50
plt.plot(validation_steps, val_losses, label="val loss")

plt.xlabel("steps")
plt.ylabel("loss")
plt.legend()
plt.show()